In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import glob
import numpy as np
import torch
import torchaudio
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor
from sklearn.metrics import classification_report, f1_score
from collections import Counter
from tqdm.auto import tqdm

BASE_DIR = os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__"))))

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct")
model = Qwen2AudioForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-Audio-7B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.eval()

TARGET_SR = processor.feature_extractor.sampling_rate
MAX_SAMPLES = 30 * TARGET_SR
print(f"Model loaded. SR={TARGET_SR}")

/data/liharrison/miniconda3/envs/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 876/876 [00:19<00:00, 44.50it/s, Materializing param=multi_modal_projector.linear.weight]                           


Model loaded. SR=16000


In [2]:
SYSTEM_PROMPT = "You are an expert speech-language pathologist."


def get_prompt(counterbalance: bool = None):
    if counterbalance is None:
        counterbalance = random.random() < 0.5

    control_desc = (
        "CONTROL — Speech is fluent with no pathological features. "
        "Normal hesitations, filler words, self-corrections, and brief pauses "
        "may be present but are typical of healthy speech.\n"
        "    Example: aɪ səpˈoʊz ðə tɹˈɪp ðæt stˈeɪz wɪð mˌiː mˈoʊst wʌz, "
        "wˈɛl, ɐ kwˈaɪət sˈʌmɚ wiː spˈɛnt ˌʌp æt lˈeɪk plˈæsɪd"
    )
    lvppa_desc = (
        "lvPPA — Speech shows features of logopenic variant primary progressive aphasia, "
        "including abnormally prolonged pauses during word retrieval, "
        "phonological errors (sound-level distortions, substitutions, or false starts), "
        "repetitive attempts at words, and fragmented sentence production.\n"
        "    Example: aɪ səpˈoʊ[PROLONG]z ðə, ðə, ðə θˈɪŋ wiː dˈɪd, "
        "ðə tɹˈɪp ðæt stˈeɪz wɪð mˌiː mˈoʊst, ɪt wʌzɐ kwˈaɪ...kwˈaɪət"
    )

    if counterbalance:
        a_desc, b_desc = control_desc, lvppa_desc
        a_label, b_label = "CONTROL", "lvPPA"
    else:
        a_desc, b_desc = lvppa_desc, control_desc
        a_label, b_label = "lvPPA", "CONTROL"

    prompt = (
        "Listen carefully to this audio of a person speaking.\n\n"
        "Which best describes this speaker?\n"
        f"A. {a_desc}\n\n"
        f"B. {b_desc}\n\n"
        "Respond with only the letter: A or B."
    )

    option_map = {"A": a_label, "B": b_label}
    return prompt, option_map


def classify_audio(audio_path):
    wav, sr = torchaudio.load(audio_path)
    wav = torchaudio.functional.resample(wav, sr, TARGET_SR).mean(0)
    if wav.shape[0] > MAX_SAMPLES:
        wav = wav[:MAX_SAMPLES]
    audio_np = wav.numpy()

    prompt, option_map = get_prompt()

    conversation = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [
            {"type": "audio", "audio_url": audio_path},
            {"type": "text", "text": prompt},
        ]},
    ]
    text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text, audio=[audio_np], sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=10)
    ids = ids[:, inputs["input_ids"].size(1):]
    response = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
    return response, option_map


def parse_response(response, option_map):
    r = response.strip().upper()
    # Check for letter answer
    if r.startswith("A"):
        return 1 if option_map["A"] == "lvPPA" else 0
    if r.startswith("B"):
        return 1 if option_map["B"] == "lvPPA" else 0
    # Fallback: scan for keywords
    r_lower = response.lower()
    if "lvppa" in r_lower or "dysfluent" in r_lower:
        return 1
    if any(w in r_lower for w in ["control", "healthy", "normal", "fluent"]):
        return 0
    return -1


import random
random.seed(42)
print("Inference functions ready.")

Inference functions ready.


In [3]:
import IPython.display as ipd

# Quick sanity check on one control clip
test_file = "/data/liharrison/lvsim/data/real/CapiloutoCCSeg-20260227T114756Z-1-001/CapiloutoCCSeg/capilouto03a_PAR_032.wav"
print(f"File: {os.path.basename(test_file)}")

# Play it
wav, sr = torchaudio.load(test_file)
wav = torchaudio.functional.resample(wav, sr, TARGET_SR).mean(0)
if wav.shape[0] > MAX_SAMPLES:
    wav = wav[:MAX_SAMPLES]
ipd.display(ipd.Audio(wav.numpy(), rate=TARGET_SR))

# First, ask what the audio contains (no classification bias)
conversation = [
    {"role": "user", "content": [
        {"type": "audio", "audio_url": test_file},
        {"type": "text", "text": "What do you hear in this audio? Describe it briefly."},
    ]},
]
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
inputs = processor(text=text, audio=[wav.numpy()], sampling_rate=TARGET_SR, return_tensors="pt", padding=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
with torch.no_grad():
    ids = model.generate(**inputs, max_new_tokens=100)
ids = ids[:, inputs["input_ids"].size(1):]
desc = processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
print(f"\nDescription: {desc}")

# Now classify
resp, option_map = classify_audio(test_file)
print(f"\nFull response: {resp}")
print(f"Option map: {option_map}")
parsed = parse_response(resp, option_map)
print(f"Parsed: {'dysfluent' if parsed == 1 else 'healthy' if parsed == 0 else 'unparsed'}")

File: capilouto03a_PAR_032.wav



Description: I heard a story read out loud, consisting of words that describe a scene where a prince sends someone to try a glass slipper on every girl's foot in the fairy tale 'The Wicked Stepmother'.

Full response: B
Option map: {'A': 'lvPPA', 'B': 'CONTROL'}
Parsed: healthy


In [4]:
REAL_DIR = os.path.join(BASE_DIR, "data", "real")
groups = {
    "lvPPA":     (sorted(glob.glob(os.path.join(REAL_DIR, "*lvPPA*", "**", "*.wav"), recursive=True)), 1),
    "JHU":       (sorted(glob.glob(os.path.join(REAL_DIR, "*jhu*", "**", "*.wav"), recursive=True)), 1),
    "Control":   (sorted(glob.glob(os.path.join(REAL_DIR, "*segmentedcc*", "**", "*.wav"), recursive=True)), 0),
    "Capilouto": (sorted(glob.glob(os.path.join(REAL_DIR, "*Capilouto*", "**", "*.wav"), recursive=True)), 0),
}

all_true, all_pred, all_raw = [], [], []

for name, (files, true_label) in groups.items():
    preds, raw_responses = [], []
    for f in tqdm(files, desc=name):
        resp, option_map = classify_audio(f)
        raw_responses.append(resp)
        pred = parse_response(resp, option_map)
        if pred == -1:
            pred = 0
        preds.append(pred)

    correct = sum(1 for p in preds if p == true_label)
    total = len(preds)
    acc = correct / total if total > 0 else 0
    dys_count = sum(1 for p in preds if p == 1)

    all_true.extend([true_label] * total)
    all_pred.extend(preds)
    all_raw.extend(raw_responses)

    print(
        f"{name:>10s}  ({total:3d} clips)  acc={acc:.3f}  "
        f"dys={dys_count}  healthy={total - dys_count}"
    )

lvPPA: 100%|██████████| 89/89 [00:20<00:00,  4.29it/s]


     lvPPA  ( 89 clips)  acc=0.787  dys=70  healthy=19


JHU: 100%|██████████| 74/74 [00:17<00:00,  4.34it/s]


       JHU  ( 74 clips)  acc=0.662  dys=49  healthy=25


Control: 100%|██████████| 235/235 [00:55<00:00,  4.25it/s]


   Control  (235 clips)  acc=0.438  dys=132  healthy=103


Capilouto: 100%|██████████| 311/311 [01:14<00:00,  4.17it/s]

 Capilouto  (311 clips)  acc=0.315  dys=213  healthy=98


In [23]:
all_true = np.array(all_true)
all_pred = np.array(all_pred)

print("=" * 60)
print("Overall Classification Report (Zero-Shot Qwen2-Audio)")
print("=" * 60)
print(classification_report(all_true, all_pred, target_names=["healthy", "dysfluent"]))
print(f"F1 Macro: {f1_score(all_true, all_pred, average='macro'):.4f}")
print(f"Accuracy: {(all_true == all_pred).mean():.4f}")

print("\nRaw response distribution:")
for resp, count in Counter(all_raw).most_common(15):
    print(f"  '{resp}': {count}")

Overall Classification Report (Zero-Shot Qwen2-Audio)
              precision    recall  f1-score   support

     healthy       0.74      0.26      0.38       546
   dysfluent       0.22      0.69      0.33       163

    accuracy                           0.36       709
   macro avg       0.48      0.47      0.36       709
weighted avg       0.62      0.36      0.37       709

F1 Macro: 0.3571
Accuracy: 0.3583

Raw response distribution:
  'B': 546
  'A': 163
